# Model Comparison — AutoML Algorithm Recommendation System
Loads `results/model_metrics.json` and produces a grouped bar chart comparing all models.

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

METRICS_PATH = os.path.join('..', 'results', 'model_metrics.json')

with open(METRICS_PATH) as f:
    data = json.load(f)

print('Loaded metrics for:', list(data['model_results'].keys()))

In [ ]:
# Build structured DataFrame
records = []
for model_name, values in data['model_results'].items():
    records.append({
        'Model':     model_name,
        'Accuracy':  round(values['accuracy'],  4),
        'Precision': round(values['precision'], 4),
        'Recall':    round(values['recall'],    4),
        'F1-Score':  round(values['f1_score'],  4),
        'CV Score':  round(values['best_cv_score'], 4),
    })

df = pd.DataFrame(records).set_index('Model')
print(df.to_string())

In [ ]:
# Grouped bar chart
metrics  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
plot_df  = df[metrics].reset_index().melt(id_vars='Model', var_name='Metric', value_name='Score')

sns.set_theme(style='whitegrid', font_scale=1.1)
fig, ax = plt.subplots(figsize=(11, 6))

sns.barplot(
    data=plot_df,
    x='Metric', y='Score', hue='Model',
    palette='Set2', edgecolor='white', ax=ax
)

# Annotate bars
for bar in ax.patches:
    h = bar.get_height()
    if h > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.005,
            f'{h:.3f}',
            ha='center', va='bottom', fontsize=8.5
        )

ax.set_title('Model Comparison — Accuracy / Precision / Recall / F1-Score',
             fontsize=14, fontweight='bold', pad=14)
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'model_comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/model_comparison_chart.png')

In [ ]:
# Best model summary
best_name = data['best_model']['name']
best_acc  = data['best_model']['accuracy']
print(f'Best Model : {best_name}')
print(f'Accuracy   : {best_acc:.4f}')
print()
print(df.loc[best_name].to_string())